In [5]:
from pathlib import Path
import json

RAW_ROOT = Path("/mnt/c/Users/MSI/Desktop/EEG_FYP/data/raw")
RESULTS_DIR = Path("/mnt/c/Users/MSI/Desktop/EEG_FYP/results")
RESULTS_DIR.mkdir(exist_ok=True)

records_file = RAW_ROOT / "RECORDS-WITH-SEIZURES"

seizure_files_relative = []
seizure_files_full = []
patient_counts = {}

with open(records_file, "r", errors="ignore") as f:
    for line in f:
        line = line.strip()

        if not line or not line.endswith(".edf"):
            continue

        seizure_files_relative.append(line)

        full_path = RAW_ROOT / line
        seizure_files_full.append(str(full_path))

        patient = line.split("/")[0]
        patient_counts[patient] = patient_counts.get(patient, 0) + 1


print("=" * 60)
print("SEIZURE FILE COUNT SUMMARY")
print("=" * 60)

for patient in sorted(patient_counts.keys()):
    print(f"{patient}: {patient_counts[patient]} seizure EDF files")

print("=" * 60)
print("Total seizure EDF files:", len(seizure_files_relative))
print("=" * 60)


output_json = RESULTS_DIR / "seizure_files_from_records.json"

data_to_save = {
    "raw_root": str(RAW_ROOT),
    "total_seizure_files": len(seizure_files_relative),
    "patient_counts": patient_counts,
    "relative_paths": seizure_files_relative,
    "full_paths": seizure_files_full
}

with open(output_json, "w") as f:
    json.dump(data_to_save, f, indent=4)

print("Saved JSON to:", output_json)

SEIZURE FILE COUNT SUMMARY
chb01: 7 seizure EDF files
chb02: 3 seizure EDF files
chb03: 7 seizure EDF files
chb04: 3 seizure EDF files
chb05: 5 seizure EDF files
chb06: 7 seizure EDF files
chb07: 3 seizure EDF files
chb08: 5 seizure EDF files
chb09: 3 seizure EDF files
chb10: 7 seizure EDF files
chb11: 3 seizure EDF files
chb12: 13 seizure EDF files
chb13: 8 seizure EDF files
chb14: 7 seizure EDF files
chb15: 14 seizure EDF files
chb16: 6 seizure EDF files
chb17: 3 seizure EDF files
chb18: 6 seizure EDF files
chb19: 3 seizure EDF files
chb20: 6 seizure EDF files
chb21: 4 seizure EDF files
chb22: 3 seizure EDF files
chb23: 3 seizure EDF files
chb24: 12 seizure EDF files
Total seizure EDF files: 141
Saved JSON to: /mnt/c/Users/MSI/Desktop/EEG_FYP/results/seizure_files_from_records.json


In [6]:
from pathlib import Path

PROCESSED_ROOT = Path("/mnt/d/EEG_FYP_DATA/processed")

all_npz = list(PROCESSED_ROOT.rglob("*.npz"))

print("Total npz files found:", len(all_npz))

for group in ["group1", "group2", "group3", "group4"]:
    files = list((PROCESSED_ROOT / group).rglob("*.npz"))
    print(group, "npz files:", len(files))

Total npz files found: 660
group1 npz files: 329
group2 npz files: 119
group3 npz files: 38
group4 npz files: 174


In [7]:
from config import SPH_MIN, SOP_MIN, WINDOW_SIZE_SEC, OVERLAP_SEC

print("SPH_MIN:", SPH_MIN)
print("SOP_MIN:", SOP_MIN)
print("WINDOW_SIZE_SEC:", WINDOW_SIZE_SEC)
print("OVERLAP_SEC:", OVERLAP_SEC)

SPH_MIN: 1
SOP_MIN: 5
WINDOW_SIZE_SEC: 5
OVERLAP_SEC: 2.5


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/mnt/c/Users/MSI/Desktop/EEG_FYP")
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))
from utils import get_final17_prediction_file_lists, build_dataset_balance_plan

# load files
train_files, val_files, test_files = get_final17_prediction_file_lists()

# rebuild balance plan
balance_plan = build_dataset_balance_plan(
    train_files,
    train_ratio=4
)

print("NEW balance plan:")
print(balance_plan)

KeyboardInterrupt: 

In [1]:
from utils import get_final17_prediction_file_lists, build_dataset_balance_plan

train_files, val_files, test_files = get_final17_prediction_file_lists()

balance_plan = build_dataset_balance_plan(train_files, train_ratio=4)

print(balance_plan)


KeyboardInterrupt



In [ ]:
from pathlib import Path
import sys
import numpy as np


sys.path.append(str(Path("../src").resolve()))

from config import (
    FINAL_17_CHANNELS,
    WINDOW_SIZE_SEC,
    TARGET_SFREQ
)

from utils import (
    get_final17_prediction_file_lists,
    prediction_batch_generator,
    prediction_eval_generator,
    build_dataset_balance_plan,
    print_dataset_balance_plan,
     transpose_generator
     
)

import json
import time
from pathlib import Path
import numpy as np

from utils import (
    prediction_batch_generator,
    transpose_generator,
    get_final17_prediction_file_lists
)

from pathlib import Path

PROJECT_ROOT = Path("/mnt/c/Users/MSI/Desktop/EEG_FYP")

results_dir = PROJECT_ROOT / "results"
cache_dir = PROJECT_ROOT / "data" / "cached_train_batches"
cache_dir.mkdir(parents=True, exist_ok=True)

#  use real train files
train_files, val_files, test_files = get_final17_prediction_file_lists()
print("Train files:", len(train_files))

# load saved balance plan
with open(results_dir / "balance_plan_train_ratio_4.json", "r") as f:
    balance_plan = json.load(f)

# settings
batch_size = 8
num_batches_to_cache = 200

# resume logic
existing_batches = sorted(cache_dir.glob("train_batch_*.npz"))
start_index = len(existing_batches)

print("Existing cached batches:", start_index)
print("New batches to add:", num_batches_to_cache)

# generator
train_generator = prediction_batch_generator(
    file_paths=train_files,
    batch_size=batch_size,
    balance_plan=balance_plan,
    preictal_fraction=0.25,
    shuffle=True,
    random_state=42 + start_index
)

train_generator = transpose_generator(train_generator)

print("Creating cached batches...")
start_time = time.time()

for i in range(num_batches_to_cache):
    batch_index = start_index + i

    X_batch, y_batch = next(train_generator)

    save_path = cache_dir / f"train_batch_{batch_index:05d}.npz"

    np.savez(
        save_path,
        X=X_batch.astype(np.float32),
        y=y_batch.astype(np.int64)
    )

    if i % 20 == 0:
        print(f"[{i}/{num_batches_to_cache}] saved")

print("DONE")
print("Total batches now:", len(list(cache_dir.glob("train_batch_*.npz"))))
print(f"Time: {(time.time() - start_time)/60:.2f} min")